# Guilin akustik rulman veri seti — indirme, karşılaştırma, WAV'a dönüştürme

Veri: *A multi-condition acoustic dataset of ball bearings for fault diagnosis* (Mendeley, 2026, CC BY 4.0).
5 alt veri seti (normal, dış bilezik, iç bilezik, kafes kırığı, bileşik iç-dış), her biri 9 koşul
(800/1000/1200 rpm × %0/15/30 yük) × 5 adet 10 dakikalık `.bkc` dosyası = 45 dosya, ~10,6 GB.

Bu not defteri:
1. Depoyu klonlar, Google Drive'ı bağlar.
2. **Karşılaştırma:** 1000 rpm / yüksüz koşulda normal, dış bilezik ve iç bilezik kayıtlarını karşılaştırır.
3. **Dönüştürme:** 225 dosyayı indirir, kanal 1'i (B&K 4966, Pa) 10 sn'lik 16-bit WAV kliplerine çevirir,
   `.bkc`'yi siler; her etiketi tek zip olarak Drive'a (`MyDrive/guilin/`) kopyalar. Kaldığı yerden devam eder.

Çalışma zamanı: CPU yeterli. Toplam çıktı ~8,6 GB.

In [ ]:
BRANCH = 'askinomer/inconnu'
!rm -rf /content/repo && git clone -q -b {BRANCH} https://github.com/askinomer/makine-ariza-tespit.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip -q install soundfile librosa
!df -h /content | tail -1

## 2. Normal ile arızalı kaydın karşılaştırılması (1000 rpm, %0 yük, dosya 1)

In [ ]:
import sys, os
sys.path.insert(0, '/content/repo')
from pathlib import Path
from tools.guilin_convert import list_remote, download

cmp_dir = Path('/content/cmp'); cmp_dir.mkdir(exist_ok=True)
want = {'normal': 'nm1000r-0-32Khz.1.bkc', 'outer': 'Ods1000r-0-32Khz.1.bkc', 'inner': 'Ids1000r-0-32Khz.1.bkc',
        'cage': 'lc1000r-0-32Khz.1.bkc'}
paths = {}
for label, name in want.items():
    e = next(f for f in list_remote(label) if f['name'] == name)
    p = cmp_dir / name
    if not p.exists():
        download(e, p)
    paths[label] = p
    print(label, p, p.stat().st_size)

In [ ]:
!python tools/guilin_compare.py --normal {paths['normal']} --fault outer={paths['outer']} --fault inner={paths['inner']} --fault cage={paths['cage']} --rpm 1000 --out /content/cmp/guilin_compare.png 2>&1 | grep -v Warning
from IPython.display import Image, display
display(Image('/content/cmp/guilin_compare.png'))

## 3. Tüm veri setini indir ve dönüştür

In [ ]:
# Drive yalnızca toplu dönüştürme için gerekir. 'mount failed' alırsanız: izin ekranında TÜM kutuları işaretleyin
# ve tekrar çalıştırın (gerekirse drive.mount(..., force_remount=True)).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, shutil
OUT = Path('/content/guilin_wav')
DRIVE = Path('/content/drive/MyDrive/guilin'); DRIVE.mkdir(parents=True, exist_ok=True)
LABELS = ['normal', 'outer', 'inner', 'cage', 'compound']

for label in LABELS:
    z = DRIVE / f'guilin_{label}.zip'
    if z.exists():
        print(f'{label}: Drive\'da zaten var ({z.stat().st_size/1e9:.2f} GB), atlanıyor'); continue
    r = subprocess.run([sys.executable, 'tools/guilin_convert.py', '--out', str(OUT), '--labels', label, '--workers', '4'],
                       capture_output=True, text=True)
    print('\n'.join(r.stdout.splitlines()[-4:])); print(r.stderr[-2000:])
    if r.returncode: raise SystemExit(f'{label} dönüştürülemedi')
    tmpzip = Path(f'/content/guilin_{label}.zip')
    subprocess.run(['zip', '-q', '-0', '-r', str(tmpzip), label, f'manifest_{label}.csv'], cwd=OUT, check=True)
    shutil.copy(tmpzip, z); tmpzip.unlink()
    print(f'{label}: {z} ({z.stat().st_size/1e9:.2f} GB)')

In [ ]:
import pandas as pd
m = pd.concat([pd.read_csv(OUT / f'manifest_{l}.csv') for l in LABELS if (OUT / f'manifest_{l}.csv').exists()])
print(len(m), 'klip; kırpılan örnek toplamı', m.n_clipped.sum())
print(m.groupby(['label', 'rpm', 'load']).size().unstack(['load']))
print(m.groupby('label')[['rms_pa', 'peak_pa']].describe().round(4))
print(m.groupby(['label','source_file']).start_utc.first().groupby('label').agg(['min','max']))
!ls -la /content/drive/MyDrive/guilin